# Nigeria events, 1960 – early 2000s

This notebook pulls event data about Nigeria from several sources and layers them by period, since no single source covers the whole span:

| Period | Source | What you get | Access |
|---|---|---|---|
| 1960 → | **NYT Archive API** | Article metadata (headline, abstract, keywords) | Free API key |
| 1960 → | **Wikidata SPARQL** | Curated list of notable events (coups, elections, wars) | No key |
| 1979 → | **GDELT 1.0** | Machine-coded events ("who did what to whom, where") | No key (bulk files) |
| 1946 → / 1989 → | **UCDP** | Conflict-years (no key) and georeferenced violent events (token) | Token for event API |
| 1997 → | **ACLED** | Political violence and protest events | Account + approved API access |

**Before you start, get the credentials you need** (you can skip any section whose credentials you don't have; every section is independent):

- **NYT:** create an app at https://developer.nytimes.com and enable the Archive API.
- **UCDP:** tokens are free but not self-service. Email the maintainer as described at https://ucdp.uu.se/apidocs (replies take a few working days). The conflict-level section works without a token.
- **ACLED:** register at https://acleddata.com. API access may need to be enabled for your account separately; until then data requests return HTTP 403.

Set credentials as environment variables (recommended) or paste them into the config cell. Every section caches what it downloads in `data/`, so re-running is cheap and an interrupted run picks up where it left off.

In [4]:
# Run once if needed
%pip install jupyterlab pandas requests matplotlib

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (2.4 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 37.8 MB/s  0:00:00m0:00:0100:01
Using cach

## 1. Configuration

In [2]:
import os, io, json, time, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt

# --- Credentials (environment variables, or paste values here) ---
NYT_API_KEY    = os.getenv("NYT_API_KEY", "cdQgOPoBaNGGAIQR0vBMTrZCNlVJLw8XoKVVYl0pIABVqmvD")
UCDP_TOKEN     = os.getenv("UCDP_TOKEN", "")
ACLED_EMAIL    = os.getenv("ACLED_EMAIL", "")
ACLED_PASSWORD = os.getenv("ACLED_PASSWORD", "")

# --- Periods (the "practical approach") ---
NYT_YEARS      = range(1960, 1979)   # NYT covers everything, but it's slow (rate-limited); extend to 2006 if you want
WIKIDATA_START, WIKIDATA_END = "1960-01-01", "2005-12-31"
GDELT_YEARS    = range(1979, 2006)   # GDELT 1.0 publishes one zip per year for 1979–2005
UCDP_START, UCDP_END   = "1989-01-01", "2005-12-31"
ACLED_START, ACLED_END = "1997-01-01", "2005-12-31"

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "nigeria-events-notebook/1.0 (research use)"}

print("NYT key set:", bool(NYT_API_KEY), "| UCDP token set:", bool(UCDP_TOKEN),
      "| ACLED login set:", bool(ACLED_EMAIL and ACLED_PASSWORD))

NYT key set: True | UCDP token set: False | ACLED login set: False


## 2. New York Times Archive API (1960 →)

The Archive API returns **every** article for a given month, so we download month by month and keep only Nigeria-related items. Matching uses NYT's own location keywords plus a list of Nigerian places and figures from the period. Expect some false positives (e.g. "Lagos" in Portugal); the `match` column tells you why each row was kept.

The API allows roughly 5 requests per minute and 500 per day, so this cell sleeps 12 s between uncached months. 1960–1978 is 228 months, about 45 minutes the first time. Results are cached per month, so you can stop and restart.

In [3]:
NIGERIA_TERMS = [
    "nigeria", "nigerian", "biafra", "biafran", "lagos", "abuja", "kano", "kaduna", "ibadan",
    "enugu", "port harcourt", "calabar", "niger delta", "ogoni",
    "balewa", "azikiwe", "awolowo", "ironsi", "gowon", "ojukwu", "murtala", "obasanjo",
    "shagari", "buhari", "babangida", "abacha", "abiola", "abubakar", "saro-wiwa",
]

def _text(doc):
    parts = [
        (doc.get("headline") or {}).get("main") or "",
        doc.get("abstract") or "",
        doc.get("snippet") or "",
        doc.get("lead_paragraph") or "",
    ]
    return " ".join(parts).lower()

def nigeria_match(doc):
    # Return why a doc matched ('keyword:...' or 'text:...'), or None.
    for kw in doc.get("keywords") or []:
        if "nigeria" in str(kw.get("value", "")).lower() or "biafra" in str(kw.get("value", "")).lower():
            return f"keyword:{kw.get('name', '')}"
    text = _text(doc)
    for term in NIGERIA_TERMS:
        if term in text:
            return f"text:{term}"
    return None

def nyt_month(year, month):
    # Fetch one month from the Archive API (with on-disk cache of the *filtered* rows).
    cache = DATA_DIR / "nyt" / f"{year}-{month:02d}.json"
    cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        return json.loads(cache.read_text()), True

    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    for attempt in range(4):
        r = requests.get(url, params={"api-key": NYT_API_KEY}, headers=HEADERS, timeout=120)
        if r.status_code == 429:               # rate limited: back off and retry
            time.sleep(60 * (attempt + 1)); continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"NYT rate limit persisted for {year}-{month:02d}")

    docs = (r.json().get("response") or {}).get("docs") or []
    rows = []
    for d in docs:
        why = nigeria_match(d)
        if why:
            rows.append({
                "date": d.get("pub_date"),
                "headline": (d.get("headline") or {}).get("main"),
                "abstract": d.get("abstract") or d.get("snippet"),
                "section": d.get("section_name"),
                "type": d.get("type_of_material"),
                "keywords": "; ".join(f"{k.get('name')}={k.get('value')}" for k in d.get("keywords") or []),
                "url": d.get("web_url"),
                "match": why,
            })
    cache.write_text(json.dumps(rows))
    return rows, False

def fetch_nyt(years):
    all_rows = []
    for y in years:
        for m in range(1, 13):
            rows, cached = nyt_month(y, m)
            all_rows += rows
            print(f"\rNYT {y}-{m:02d}: {len(rows):4d} matches (total {len(all_rows)})", end="")
            if not cached:
                time.sleep(12)
    print()
    df = pd.DataFrame(all_rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None)
        df = df.sort_values("date").reset_index(drop=True)
    return df

if NYT_API_KEY:
    nyt = fetch_nyt(NYT_YEARS)
    nyt.to_csv(DATA_DIR / "nyt_nigeria.csv", index=False)
    display(nyt.head(10))
else:
    nyt = pd.DataFrame()
    print("Skipping NYT: set NYT_API_KEY.")

NYT 1978-12:    7 matches (total 3435)


,date,headline,abstract,section,type,keywords,url,match
0,1960-01-03 05:00:00,NIGERIAN WARNS PRESS; Minister Charges Critici...,Min Benson warns newsmen he will urge action a...,Archives,Archives,glocations=Nigeria; glocations=Nigeria; subjec...,https://www.nytimes.com/1960/01/03/archives/ni...,keyword:glocations
1,1960-01-12 05:00:00,NIGERIANS CHEER MACMILLAN VISIT; Independence ...,Ghana Council for Nuclear Disarmament scores him,Archives,Archives,glocations=Nigeria; glocations=Nigeria; glocat...,https://www.nytimes.com/1960/01/12/archives/ni...,keyword:glocations
2,1960-01-13 05:00:00,MACMILLAN MEETS NIGERIA'S CABINET,,Archives,Archives,glocations=Nigeria; glocations=AFRICA,https://www.nytimes.com/1960/01/13/archives/ma...,keyword:glocations
3,1960-01-14 05:00:00,Macmillan Denies Independence Of Rhodesia Fede...,"holds news conf, Lagos, Nigeria",Archives,Archives,glocations=Nigeria; glocations=NYASALAND; gloc...,https://www.nytimes.com/1960/01/14/archives/ma...,keyword:glocations
4,1960-01-15 05:00:00,"NIGERIANS ASK LIBERTY; 1,200 Ibadan Students R...",,Archives,Archives,glocations=Nigeria; glocations=AFRICA,https://www.nytimes.com/1960/01/15/archives/ni...,keyword:glocations
5,1960-01-17 05:00:00,SOVEREIGNTY STEP TAKEN BY NIGERIA; Parliament ...,in Nigeria,Archives,Archives,glocations=Nigeria; glocations=AFRICA; organiz...,https://www.nytimes.com/1960/01/17/archives/so...,keyword:glocations
6,1960-01-17 05:00:00,Article 1 -- No Title,further activities,Archives,Archives,glocations=Nigeria; subject=MACMILLAN VISIT,https://www.nytimes.com/1960/01/17/archives/ar...,keyword:glocations
7,1960-01-24 05:00:00,Anthem Strikes Wrong Note,new natl anthem selected after contest found t...,Archives,Archives,glocations=Nigeria,https://www.nytimes.com/1960/01/24/archives/an...,keyword:glocations
8,1960-01-30 05:00:00,27 Nigerian Tribesmen Accused Of Joining in Ca...,27 Izi tribesmen jailed for suspected cannibal...,Archives,Archives,glocations=Nigeria,https://www.nytimes.com/1960/01/30/archives/27...,keyword:glocations
9,1960-01-31 05:00:00,BARBARIAN CULT FEARED IN NIGERIA; Police Facin...,lr on article; ed note,Archives,Archives,glocations=Nigeria; subject=ORGANIZATIONS AND ...,https://www.nytimes.com/1960/01/31/archives/ba...,keyword:glocations


In [ ]:
# Optional: keep only the strongest matches (NYT tagged the article with a Nigeria location/subject)
if not nyt.empty:
    nyt_strict = nyt[nyt["match"].str.startswith("keyword")]
    print(len(nyt), "matches total;", len(nyt_strict), "tagged by NYT itself")
    display(nyt.groupby(nyt["date"].dt.year).size().rename("articles").to_frame().T)

## 3. Wikidata SPARQL (1960 →)

Wikidata gives a curated list of notable events: anything whose country is Nigeria and that has a point-in-time or start date in range. It's small and fast, which makes it a good backbone timeline to read the noisier sources against.

In [4]:

WIKIDATA_QUERY = '''
SELECT DISTINCT ?event ?eventLabel ?date ?typeLabel ?description ?article WHERE {
  ?event wdt:P17 wd:Q1033 ;          # country = Nigeria
         wdt:P31 ?type .
  { ?event wdt:P585 ?date } UNION { ?event wdt:P580 ?date }   # point in time / start time
  FILTER(?date >= "%s"^^xsd:dateTime && ?date <= "%s"^^xsd:dateTime)
  FILTER NOT EXISTS { ?event wdt:P31 wd:Q5 }                   # exclude people
  OPTIONAL { ?event schema:description ?description FILTER(LANG(?description) = "en") }
  OPTIONAL { ?article schema:about ?event ; schema:isPartOf <https://en.wikipedia.org/> }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?date
'''

def fetch_wikidata(start, end):
    q = WIKIDATA_QUERY % (f"{start}T00:00:00Z", f"{end}T23:59:59Z")
    r = requests.get("https://query.wikidata.org/sparql",
                     params={"query": q, "format": "json"}, headers=HEADERS, timeout=120)
    r.raise_for_status()
    rows = [{k: v["value"] for k, v in b.items()} for b in r.json()["results"]["bindings"]]
    # Optional fields are absent from rows that lack them, so make sure every column exists
    df = pd.DataFrame(rows, columns=["event", "eventLabel", "date", "typeLabel", "description", "article"])
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"].str[:10], errors="coerce")
    # One row per event, with all its types joined
    df = (df.groupby(["event", "eventLabel", "date"], as_index=False)
            .agg(type=("typeLabel", lambda s: "; ".join(sorted(set(s)))),
                 description=("description", "first"),
                 article=("article", "first")))
    return df.sort_values("date").reset_index(drop=True)

wikidata = fetch_wikidata(WIKIDATA_START, WIKIDATA_END)
wikidata.to_csv(DATA_DIR / "wikidata_nigeria.csv", index=False)
print(len(wikidata), "events")
display(wikidata.head(20))

466 events


,event,eventLabel,date,type,description,article
0,http://www.wikidata.org/entity/Q2583400,Nigeria at the 1960 Summer Olympics,1960-01-01,Olympic delegation,sporting event delegation,https://en.wikipedia.org/wiki/Nigeria_at_the_1...
1,http://www.wikidata.org/entity/Q927304,University of Nigeria,1960-01-01,open-access publisher; university,"federal university in Nsukka, Enugu State, Nig...",https://en.wikipedia.org/wiki/University_of_Ni...
2,http://www.wikidata.org/entity/Q20312406,1960 in Nigeria,1960-01-01,events in a specific year or time period,overview of Nigeria-related events during the ...,https://en.wikipedia.org/wiki/1960_in_Nigeria
3,http://www.wikidata.org/entity/Q7988226,1960 Western Region legislative election,1960-01-01,legislative election,NaN,https://en.wikipedia.org/wiki/1960_Western_Reg...
4,http://www.wikidata.org/entity/Q115526967,Independence Day,1960-10-01,oration,speech by Abubakar Tafawa Balewa at the Indepe...,NaN
5,http://www.wikidata.org/entity/Q140119414,Nigerian independence (1960),1960-10-01,event,Declaration of Nigeria’s independence from Bri...,NaN
6,http://www.wikidata.org/entity/Q1586750,Nigeria–Russia relations,1960-11-25,bilateral relation,international relations,https://en.wikipedia.org/wiki/Nigeria%E2%80%93...
7,http://www.wikidata.org/entity/Q20312420,1961 in Nigeria,1961-01-01,events in a specific year or time period,overview of Nigeria-related events during the ...,https://en.wikipedia.org/wiki/1961_in_Nigeria
8,http://www.wikidata.org/entity/Q7058911,1961 Northern Region legislative election,1961-01-01,legislative election,NaN,https://en.wikipedia.org/wiki/1961_Northern_Re...
9,http://www.wikidata.org/entity/Q5330406,1961 Eastern Region legislative election,1961-11-16,legislative election,NaN,https://en.wikipedia.org/wiki/1961_Eastern_Reg...


## 5. UCDP (conflict-years from 1946; georeferenced events from 1989)

Two parts:

1. **Conflict-years (no token needed):** the UCDP/PRIO Armed Conflict Dataset, downloaded as CSV. This covers the Biafran war and other state-based conflicts back to independence.
2. **Georeferenced events (token needed):** individual violent events with locations and fatality estimates, 1989 →, via the API. Nigeria's country code in UCDP is 475 (Gleditsch–Ward).

Check https://ucdp.uu.se/apidocs and https://ucdp.uu.se/downloads for the current version numbers; update them below if a newer release is out.

In [ ]:
UCDP_GED_VERSION = "26.1"
UCDP_ACD_URL = "https://ucdp.uu.se/downloads/ucdpprio/ucdp-prio-acd-261-csv.zip"
NIGERIA_GW = 475

def fetch_ucdp_conflicts():
    out = DATA_DIR / "ucdp_acd.csv"
    if not out.exists():
        r = requests.get(UCDP_ACD_URL, headers=HEADERS, timeout=120)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
            out.write_bytes(z.read(name))
    acd = pd.read_csv(out)
    is_ng = (acd["location"].astype(str).str.contains("Nigeria", case=False)
             | acd["gwno_loc"].astype(str).str.split(",").apply(lambda xs: str(NIGERIA_GW) in [x.strip() for x in xs]))
    return acd[is_ng].sort_values("year").reset_index(drop=True)

ucdp_conflicts = fetch_ucdp_conflicts()
cols = [c for c in ["conflict_id", "year", "location", "side_a", "side_b", "incompatibility",
                    "territory_name", "intensity_level", "type_of_conflict", "start_date"] if c in ucdp_conflicts]
display(ucdp_conflicts[cols])

In [ ]:
def fetch_ucdp_ged(start, end, pagesize=1000):
    url = f"https://ucdpapi.pcr.uu.se/api/gedevents/{UCDP_GED_VERSION}"
    params = {"pagesize": pagesize, "Country": NIGERIA_GW, "StartDate": start, "EndDate": end}
    hdrs = {**HEADERS, "x-ucdp-access-token": UCDP_TOKEN}
    rows, page = [], 0
    while True:
        r = requests.get(url, params={**params, "page": page}, headers=hdrs, timeout=120)
        if r.status_code == 401:
            raise RuntimeError("UCDP returned 401: token missing or invalid (see https://ucdp.uu.se/apidocs).")
        r.raise_for_status()
        js = r.json()
        rows += js.get("Result", [])
        total_pages = js.get("TotalPages", 1)
        print(f"\rUCDP GED page {page + 1}/{total_pages}", end="")
        page += 1
        if page >= total_pages or not js.get("Result"):
            break
    print()
    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date_start"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)
    return df

if UCDP_TOKEN:
    ucdp_ged = fetch_ucdp_ged(UCDP_START, UCDP_END)
    ucdp_ged.to_csv(DATA_DIR / "ucdp_ged_nigeria.csv", index=False)
    cols = [c for c in ["date", "type_of_violence", "conflict_name", "side_a", "side_b",
                        "adm_1", "where_coordinates", "best", "source_headline"] if c in ucdp_ged]
    display(ucdp_ged[cols].head(10))
else:
    ucdp_ged = pd.DataFrame()
    print("Skipping UCDP GED: set UCDP_TOKEN.")

In [5]:
import pandas as pd
from pathlib import Path

SCAD_ETYPES = {1: "Organized demonstration", 2: "Spontaneous demonstration", 3: "Organized violent riot",
               4: "Spontaneous violent riot", 5: "General strike", 6: "Limited strike",
               7: "Pro-government violence", 8: "Anti-government violence",
               9: "Extra-government violence", 10: "Intra-government violence"}

scad = pd.read_csv("data/scad/SCAD2018Africa_Final_nigeria.csv", encoding="latin-1", low_memory=False)
y = pd.to_numeric(scad["styr"], errors="coerce")
m = pd.to_numeric(scad["stmo"], errors="coerce").where(lambda s: s.between(1, 12), 1)
d = pd.to_numeric(scad["stday"], errors="coerce").where(lambda s: s.between(1, 31), 1)
scad["date"] = pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")
scad = scad[scad["date"].dt.year.between(1990, 2005)].sort_values("date").reset_index(drop=True)
scad["event_type"] = pd.to_numeric(scad["etype"], errors="coerce").map(SCAD_ETYPES)
scad["ndeath"] = pd.to_numeric(scad["ndeath"], errors="coerce").where(lambda s: s >= 0)

print(len(scad), "events")
with pd.option_context("display.max_colwidth", 200):
    display(scad[["date", "countryname", "event_type", "elocal", "ndeath", "issuenote"]].head(15))

741 events


,date,countryname,event_type,elocal,ndeath,issuenote
0,1990-04-22,Nigeria,Intra-government violence,Lagos,NaN,Rebel soldiers attempt coup in order to end 'Muslim domination.'
1,1991-02-20,Nigeria,Limited strike,Nationwide,0.0,Journalists strike and several are arrested
2,1991-04-25,Nigeria,Spontaneous violent riot,Bauchi,246.0,Islamic fundamentalists attack Christians
3,1991-05-29,Nigeria,Spontaneous violent riot,Lagos,2.0,Students attempt attack on head of college and are killed by other students
4,1991-05-31,Nigeria,Limited strike,Nationwide,0.0,"Newspaper workers strike, demanding higher wages"
5,1991-06-13,Nigeria,Spontaneous violent riot,Lagos,0.0,Students kidnap children and family members of university employees after riot
6,1991-08-08,Nigeria,Spontaneous demonstration,Borno,0.0,Former oil minister goes on hunger strike to protest illegal detention
7,1991-08-29,Nigeria,Spontaneous violent riot,Nationwide,0.0,Citizens riot after creation of nine new states is announced
8,1991-10-14,Nigeria,Spontaneous demonstration,Kano,110.0,Christians and Muslims clash after Muslims protest arrival of Christian evangelist
9,1991-10-24,Nigeria,Spontaneous violent riot,Kaduna,0.0,"Students riot, protesting state government's policies on books and exams"


In [ ]:
# ---- Phoenix headlines (Cline Center MetaData files) ----
import re
import pandas as pd
from pathlib import Path

# The MetaData files to search (these hold the headlines). Set these to where you saved them.
# Leave the list empty ([]) to search this folder, data/ and Downloads for MetaData*.csv instead.
PHOENIX_META_FILES = [
    "data/phoenix/MetaDataNYT_1945-2005.csv",
    "data/phoenix/MetaDataBLN-NYT_1980-2018.csv",
    "data/phoenix/MetaDataBLN-NYT_1980-2018.csv"
    "data/phoenix/MetaDataFBIS_1995-2004.csv"
]
PHOENIX_YEARS = (1960, 2005)

try:
    DATA_DIR
except NameError:
    DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR = DATA_DIR / "phoenix_cache"; CACHE_DIR.mkdir(exist_ok=True)

try:
    NIGERIA_TERMS
except NameError:
    NIGERIA_TERMS = [
        "nigeria", "nigerian", "biafra", "biafran", "lagos", "abuja", "kano", "kaduna", "ibadan",
        "enugu", "port harcourt", "calabar", "niger delta", "ogoni",
        "balewa", "azikiwe", "awolowo", "ironsi", "gowon", "ojukwu", "murtala", "obasanjo",
        "shagari", "buhari", "babangida", "abacha", "abiola", "abubakar", "saro-wiwa",
    ]
NIGERIA_RE = re.compile(r"\b(?:" + "|".join(re.escape(t) for t in NIGERIA_TERMS) + r")", re.IGNORECASE)

def find_files(configured, pattern):
    if configured:
        return [Path(f).expanduser() for f in configured]
    found = []
    for folder in [Path("."), Path("data"), Path.home() / "Downloads"]:
        if folder.exists():
            found += [p for p in folder.rglob(pattern) if "_nigeria" not in p.stem and CACHE_DIR not in p.parents]
    return sorted(set(found))

def clean_title(s):
    return s.str.replace("';", "'", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()

def _read_meta(path, chunksize):
    # Read a Phoenix MetaData file in chunks with standard column names. Handles both layouts:
    #   NYT: aid, title, primary_source, original_source, pubdate
    #   SWB: aid, url, title, source, pubdate   (header row optional)
    first = pd.read_csv(path, nrows=0).columns
    has_header = any(str(c).strip().lower() == "title" for c in first)
    if has_header:
        reader = pd.read_csv(path, dtype=str, chunksize=chunksize, on_bad_lines="skip")
    else:
        names = ["aid", "url", "title", "primary_source", "pubdate"][:len(first)]
        reader = pd.read_csv(path, dtype=str, chunksize=chunksize, on_bad_lines="skip", header=None, names=names)
    for chunk in reader:
        chunk.columns = [str(c).strip().lower() for c in chunk.columns]
        chunk = chunk.rename(columns={"source": "primary_source", "link": "url"})
        if "url" not in chunk:
            # some layouts put the link in an unnamed column; pick any column holding http links
            for c in chunk.columns:
                if c not in ("aid", "title", "pubdate") and chunk[c].astype(str).str.startswith("http").any():
                    chunk = chunk.rename(columns={c: "url"}); break
        if "primary_source" not in chunk:
            chunk["primary_source"] = path.stem.replace("MetaData", "").split("_")[0]
        yield chunk

def phoenix_headlines(path, years, chunksize=500_000):
    cache = CACHE_DIR / f"{path.stem}_nigeria.csv"
    if cache.exists():
        df = pd.read_csv(cache, dtype=str)
    else:
        parts = [c[c["title"].fillna("").str.contains(NIGERIA_RE)] for c in _read_meta(path, chunksize)]
        df = pd.concat(parts, ignore_index=True)
        df.to_csv(cache, index=False)
    df["title"] = clean_title(df["title"].fillna(""))
    df["date"] = pd.to_datetime(df["pubdate"], format="%m/%d/%Y", errors="coerce")
    df["file"] = path.name
    return df[df["date"].dt.year.between(*years)]

candidates = find_files(PHOENIX_META_FILES, "MetaData*.csv")
for p in candidates:
    if not p.exists():
        print("Not found, skipping:", p)
meta_files = [p for p in candidates if p.exists()]

if meta_files:
    phoenix = pd.concat([phoenix_headlines(p, PHOENIX_YEARS) for p in meta_files], ignore_index=True)
    phoenix = phoenix.drop_duplicates(["aid", "primary_source"])
    # Many stories are listed twice under different IDs; drop repeats of the same source, date and headline
    key = (phoenix["primary_source"] + "|" + phoenix["date"].astype(str) + "|"
           + phoenix["title"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True).str[:25])
    phoenix = phoenix[~key.duplicated()].sort_values("date").reset_index(drop=True)
    phoenix.to_csv(DATA_DIR / "phoenix_headlines_nigeria.csv", index=False)

    print("Files:", ", ".join(p.name for p in meta_files))
    print(f"{len(phoenix):,} Nigeria headlines, {PHOENIX_YEARS[0]}–{PHOENIX_YEARS[1]}")
    display(phoenix.groupby([phoenix["date"].dt.year // 10 * 10, "primary_source"]).size()
                   .unstack(fill_value=0).rename_axis("decade"))
    with pd.option_context("display.max_colwidth", 150):
        display(phoenix[["date", "primary_source", "title"]].head(20))
else:
    phoenix = pd.DataFrame()
    print("No MetaData files found. Check the paths in PHOENIX_META_FILES.")

Not found, skipping: data/phoenix/MetaDataBLN-NYT_1980-2018.csvdata/phoenix/MetaDataFBIS_1995-2004.csv
Files: MetaDataNYT_1945-2005.csv, MetaDataBLN-NYT_1980-2018.csv
1,364 Nigeria headlines, 1960–2005


primary_source,NYT
decade,
1960,414
1970,182
1980,240
1990,344
2000,184


,date,primary_source,title
0,1960-01-03,NYT,NIGERIAN WARNS PRESS Minister Charges Criticism Is Defaming New State
1,1960-01-13,NYT,MACMILLAN MEETS NIGERIA'S CABINET
2,1960-01-17,NYT,SOVEREIGNTY STEP TAKEN BY NIGERIA Parliament Agrees to Seek Independence in October -- Macmillan Tours Colony
3,1960-02-06,NYT,GRAHAM VISITS MOSLEM Evangelist Explains His Work to Nigerian Leader
4,1960-02-09,NYT,GRAHAM DEBATES MOSLEM TO DRAW Evangelist Defends Missions Against Nigerian's Charge of Political Intervention
5,1960-02-10,NYT,Nigerian to Head College
6,1960-02-12,NYT,Northern Nigeria's Chief Warns Missionaries to Avoid Politics; Sardauna of Sokoto Stresses Need for Unity -- Reveals Plans for Trip to U.S.
7,1960-02-17,NYT,NIGERIANS BATTLE SLEEPING DISEASE Cut Huge Swaths in Bush to Stem Spread of Illness Caused by Tsetse Fly
8,1960-03-05,NYT,Nigeria Names Envoy to U.S.
9,1960-03-10,NYT,Japanese Mission in Nigeria


## 7. Combined timeline

Everything is normalized into one table (`date`, `source`, `title`, `detail`, `location`, `url`) so you can search or filter across sources. GDELT is summarized differently because of its volume: only root events with at least 5 mentions are included, with a generated title.

In [ ]:
def _norm(df, source, title, detail=None, location=None, url=None):
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "source", "title", "detail", "location", "url"])
    get = lambda c: df[c] if c is not None and c in df else None
    return pd.DataFrame({
        "date": pd.to_datetime(df["date"]),
        "source": source,
        "title": get(title),
        "detail": get(detail),
        "location": get(location),
        "url": get(url),
    })

g = gdelt[(gdelt["IsRootEvent"] == "1") & (gdelt["NumMentions"] >= 5)].copy() if not gdelt.empty else gdelt
if not g.empty:
    g["title"] = (g["Actor1Name"].fillna("?") + " — " + g["event_type"].fillna(g["EventCode"])
                  + " — " + g["Actor2Name"].fillna("?"))
    g["detail"] = "CAMEO " + g["EventCode"] + ", Goldstein " + g["GoldsteinScale"].astype(str) + ", mentions " + g["NumMentions"].astype(str)

if not ucdp_ged.empty:
    ucdp_ged["detail"] = ucdp_ged["side_a"] + " vs " + ucdp_ged["side_b"] + ", best estimate deaths: " + ucdp_ged["best"].astype(str)

timeline = pd.concat([
    _norm(nyt, "NYT", "headline", "abstract", None, "url"),
    _norm(wikidata, "Wikidata", "eventLabel", "description", None, "article"),
    _norm(g, "GDELT", "title", "detail", "ActionGeo_FullName"),
    _norm(ucdp_ged, "UCDP GED", "conflict_name", "detail", "where_coordinates"),
    _norm(acled, "ACLED", "event_type", "notes", "location"),
], ignore_index=True).sort_values("date").reset_index(drop=True)

timeline.to_csv(DATA_DIR / "nigeria_timeline_combined.csv", index=False)
print(timeline["source"].value_counts())
display(timeline.head(20))

In [ ]:
# Records per year by source (separate panels because the scales differ by orders of magnitude)
counts = timeline.groupby([timeline["date"].dt.year, "source"]).size().unstack(fill_value=0)
counts = counts.reindex(range(1960, counts.index.max() + 1), fill_value=0)
counts.plot(subplots=True, kind="bar", figsize=(14, 2.2 * counts.shape[1]), sharex=True, legend=False)
plt.xlabel(""); plt.tight_layout(); plt.show()

In [ ]:
# Search across everything, e.g. a keyword within a date window
def search(term, start=None, end=None):
    t = timeline
    if start: t = t[t["date"] >= start]
    if end:   t = t[t["date"] <= end]
    hit = (t["title"].astype(str).str.contains(term, case=False, na=False)
           | t["detail"].astype(str).str.contains(term, case=False, na=False))
    return t[hit]

search("coup", "1966-01-01", "1966-12-31")